# Olist Data Quality

## HVIA Data & AI Solutions Task

**Prepared by:** Youssef Atef Tayh  
**Stage:** Data Quality

This notebook checks whether the raw Olist data contains issues that could affect later analysis.

The raw CSV files are never overwritten. Any cleaned or analysis-ready data is created as a separate copy.

## Approach

For every issue we identify, we separate:

1. **Detection** — what is present in the raw data?
2. **Decision** — should it be changed, flagged, or left unchanged?
3. **Before / After** — shown whenever a safe transformation is applied.
4. **Impact** — how many records are affected?

The goal is to avoid silently changing the dataset.

# 1. Setup

In [1]:
# Data analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File paths and display
from pathlib import Path
from IPython.display import display

# Hide unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

In [2]:
current_folder = Path.cwd()

if (current_folder / "data" / "raw").exists():
    PROJECT_ROOT = current_folder
elif (current_folder.parent / "data" / "raw").exists():
    PROJECT_ROOT = current_folder.parent
else:
    raise FileNotFoundError("Could not find the project data/raw folder.")

DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data folder: {DATA_DIR}")

Project root: E:\Interniship\HVIA - Data & AI Intern 2026\Tasks\Daily Tasks\Task 1\Olist_HVIA_Youssef_Atef_Tayh
Raw data folder: E:\Interniship\HVIA - Data & AI Intern 2026\Tasks\Daily Tasks\Task 1\Olist_HVIA_Youssef_Atef_Tayh\data\raw


# 2. Load Raw Data

In [3]:
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")
translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "translation": translation,
}

print(f"Loaded {len(tables)} tables successfully.")

Loaded 9 tables successfully.


# 3. Raw Dataset Quality Snapshot

In [4]:
quality_snapshot = pd.DataFrame([
    {
        "Table": name,
        "Rows": len(table),
        "Columns": len(table.columns),
        "Missing Cells": int(table.isna().sum().sum()),
        "Exact Duplicate Rows": int(table.duplicated().sum()),
    }
    for name, table in tables.items()
])
display(quality_snapshot)

,Table,Rows,Columns,Missing Cells,Exact Duplicate Rows
0,customers,99441,5,0,0
1,orders,99441,8,4908,0
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,145903,0
5,products,32951,9,2448,0
6,sellers,3095,4,0,0
7,geolocation,1000163,5,0,261831
8,translation,71,2,0,0


### Snapshot Interpretation

This gives the baseline before any cleaning decision is made.

# 4. Missing Values

In [5]:
missing_rows = []
for name, table in tables.items():
    for column in table.columns:
        count = int(table[column].isna().sum())
        if count > 0:
            missing_rows.append({
                "Table": name,
                "Column": column,
                "Missing Count": count,
                "Missing Percentage": round(count / len(table) * 100, 2),
            })

missing_summary = pd.DataFrame(missing_rows).sort_values(
    "Missing Percentage", ascending=False
).reset_index(drop=True)

display(missing_summary)

,Table,Column,Missing Count,Missing Percentage
0,reviews,review_comment_title,87656,88.34
1,reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
3,products,product_name_lenght,610,1.85
4,products,product_category_name,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,orders,order_delivered_carrier_date,1783,1.79
8,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


### Missing Value Interpretation

The largest missing-value groups are concentrated in optional review text, product attributes, and order lifecycle timestamps.

We do not fill these values automatically because the correct treatment depends on what the field represents.

# 5. Exact Duplicate Records

In [6]:
duplicate_summary = pd.DataFrame([
    {
        "Table": name,
        "Rows Before": len(table),
        "Exact Duplicate Rows": int(table.duplicated().sum()),
    }
    for name, table in tables.items()
])

display(duplicate_summary)

,Table,Rows Before,Exact Duplicate Rows
0,customers,99441,0
1,orders,99441,0
2,order_items,112650,0
3,payments,103886,0
4,reviews,99224,0
5,products,32951,0
6,sellers,3095,0
7,geolocation,1000163,261831
8,translation,71,0


### Duplicate Decision

Exact duplicate rows can be safely removed only when the duplicate is identical across every column.

We will demonstrate this explicitly for the geolocation table. No raw file will be overwritten.

# 6. Before: Geolocation Duplicate Check

In [7]:
geo_before = geolocation.copy()

print("Rows before:", len(geo_before))
print("Exact duplicate rows:", geo_before.duplicated().sum())
display(geo_before.head())

Rows before: 1000163
Exact duplicate rows: 261831


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


# 7. After: Remove Exact Geolocation Duplicates

In [8]:
geo_clean = geo_before.drop_duplicates().copy()

print("Rows after:", len(geo_clean))
print("Exact duplicate rows after:", geo_clean.duplicated().sum())
display(geo_clean.head())

Rows after: 738332
Exact duplicate rows after: 0


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


### Before / After Interpretation

Only exact duplicate geolocation rows were removed from the working copy.

The raw geolocation CSV remains unchanged.

This is a safe structural cleanup because the removed rows are identical across all available fields.

# 8. Order Timestamp Quality

In [9]:
order_dates = orders.copy()

timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in timestamp_columns:
    order_dates[column] = pd.to_datetime(order_dates[column], errors="coerce")

timestamp_quality = pd.DataFrame([
    {
        "Column": column,
        "Missing After Conversion": int(order_dates[column].isna().sum()),
        "Min": order_dates[column].min(),
        "Max": order_dates[column].max(),
    }
    for column in timestamp_columns
])

display(timestamp_quality)

,Column,Missing After Conversion,Min,Max
0,order_purchase_timestamp,0,2016-09-04 21:15:19,2018-10-17 17:30:18
1,order_approved_at,160,2016-09-15 12:16:38,2018-09-03 17:40:06
2,order_delivered_carrier_date,1783,2016-10-08 10:34:01,2018-09-11 19:48:28
3,order_delivered_customer_date,2965,2016-10-11 13:46:32,2018-10-17 13:22:46
4,order_estimated_delivery_date,0,2016-09-30 00:00:00,2018-11-12 00:00:00


# 9. Order Lifecycle Consistency

In [10]:
lifecycle_checks = pd.DataFrame([
    ["Approval before purchase",
     int((order_dates["order_approved_at"] < order_dates["order_purchase_timestamp"]).sum())],

    ["Carrier handoff before approval",
     int((order_dates["order_delivered_carrier_date"] < order_dates["order_approved_at"]).sum())],

    ["Customer delivery before carrier handoff",
     int((order_dates["order_delivered_customer_date"] < order_dates["order_delivered_carrier_date"]).sum())],

    ["Customer delivery before purchase",
     int((order_dates["order_delivered_customer_date"] < order_dates["order_purchase_timestamp"]).sum())],

    ["Delivery after estimated date",
     int((order_dates["order_delivered_customer_date"] > order_dates["order_estimated_delivery_date"]).sum())],
], columns=["Check", "Rows"])

display(lifecycle_checks)

,Check,Rows
0,Approval before purchase,0
1,Carrier handoff before approval,1359
2,Customer delivery before carrier handoff,23
3,Customer delivery before purchase,0
4,Delivery after estimated date,7827


### Lifecycle Interpretation

Timestamp consistency checks identify records that require investigation.

A delivery after the estimated date is not automatically a data error; it can represent a real late delivery.

Missing lifecycle timestamps can also be legitimate for orders that did not reach a later stage.

# 10. Numerical Value Checks

In [11]:
numeric_checks = pd.DataFrame([
    ["Negative item price", int((order_items["price"] < 0).sum())],
    ["Negative freight value", int((order_items["freight_value"] < 0).sum())],
    ["Negative payment value", int((payments["payment_value"] < 0).sum())],
    ["Payment installments below 1", int((payments["payment_installments"] < 1).sum())],
    ["Negative product weight", int((products["product_weight_g"] < 0).sum())],
    ["Negative product length", int((products["product_length_cm"] < 0).sum())],
    ["Negative product height", int((products["product_height_cm"] < 0).sum())],
    ["Negative product width", int((products["product_width_cm"] < 0).sum())],
], columns=["Check", "Rows"])

display(numeric_checks)

,Check,Rows
0,Negative item price,0
1,Negative freight value,0
2,Negative payment value,0
3,Payment installments below 1,2
4,Negative product weight,0
5,Negative product length,0
6,Negative product height,0
7,Negative product width,0


### Numerical Check Interpretation

Negative prices, freight values, payment values, or physical dimensions would normally require investigation.

The two payment records with installments below 1 are flagged rather than changed because the source does not provide enough evidence to decide the correct replacement value.

# 11. Review Score Check

In [12]:
review_score_summary = (
    reviews["review_score"]
    .value_counts()
    .sort_index()
    .rename_axis("Review Score")
    .reset_index(name="Reviews")
)

display(review_score_summary)

,Review Score,Reviews
0,1,11424
1,2,3151
2,3,8179
3,4,19142
4,5,57328


### Review Score Interpretation

Review scores are stored on a 1–5 scale in the observed dataset.

No score values are changed.

## Payment Installment Investigation

Two payment records were flagged because `payment_installments` is below 1. Before making any cleaning decision, the records are inspected directly. No values are changed at this stage.

In [13]:
invalid_installments = payments[
    payments["payment_installments"] < 1
].copy()

print(f"Flagged records: {len(invalid_installments):,}")
display(invalid_installments)

Flagged records: 2


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


### Payment Installment Interpretation

The flagged records are isolated for investigation. The available dataset does not provide enough context to determine the intended value with certainty, so the records remain unchanged in the raw data.

## Lifecycle Inconsistency Investigation

Timestamp checks identified records with unusual chronological relationships. We inspect a sample of these records before deciding whether any transformation is appropriate.

In [14]:
lifecycle_issues = orders[
    (
        orders["order_delivered_carrier_date"].notna()
        & orders["order_approved_at"].notna()
        & (orders["order_delivered_carrier_date"] < orders["order_approved_at"])
    )
    |
    (
        orders["order_delivered_customer_date"].notna()
        & orders["order_delivered_carrier_date"].notna()
        & (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"])
    )
].copy()

print(f"Lifecycle issue records: {len(lifecycle_issues):,}")
display(
    lifecycle_issues[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ]
    ].head(20)
)

Lifecycle issue records: 1,382


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04 00:00:00
64,688052146432ef8253587b930b01a06d,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15 00:00:00
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31 00:00:00
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31 00:00:00
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06 00:00:00
481,32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-04 16:49:21,2018-07-05 16:33:06,2018-07-05 14:50:00,2018-07-07 14:41:18,2018-07-23 00:00:00
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-24 11:32:11,2018-07-29 23:30:52,2018-07-26 14:46:00,2018-07-27 18:55:57,2018-08-06 00:00:00
585,16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 01:09:09,2018-05-07 16:52:39,2018-05-07 15:09:00,2018-05-24 00:31:18,2018-06-07 00:00:00
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04 00:00:00
817,6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-03 23:40:16,2018-07-05 16:31:26,2018-07-04 12:14:00,2018-07-05 22:52:28,2018-07-19 00:00:00


### Lifecycle Investigation Result

These records are retained because this stage is about identifying and understanding data-quality issues, not silently correcting historical records. The identified inconsistencies should be considered when interpreting delivery-related metrics later.

## Data Quality Scorecard

The scorecard summarizes the main checks performed in this stage.

In [15]:
negative_prices = int((order_items["price"] < 0).sum())
negative_freight = int((order_items["freight_value"] < 0).sum())
negative_payments = int((payments["payment_value"] < 0).sum())
invalid_installment_count = int((payments["payment_installments"] < 1).sum())
review_out_of_range = int((~reviews["review_score"].between(1, 5)).sum())

quality_scorecard = pd.DataFrame([
    ["Exact duplicate rows", "Checked", int(duplicate_summary["Exact Duplicate Rows"].sum())],
    ["Missing values", "Checked", int(missing_summary["Missing Count"].sum())],
    ["Negative item prices", "Checked", negative_prices],
    ["Negative freight values", "Checked", negative_freight],
    ["Negative payment values", "Checked", negative_payments],
    ["Invalid payment installments", "Checked", invalid_installment_count],
    ["Review scores outside 1–5", "Checked", review_out_of_range],
    ["Lifecycle inconsistencies", "Checked", len(lifecycle_issues)],
], columns=["Check", "Status", "Records Flagged"])

display(quality_scorecard)

,Check,Status,Records Flagged
0,Exact duplicate rows,Checked,261831
1,Missing values,Checked,153259
2,Negative item prices,Checked,0
3,Negative freight values,Checked,0
4,Negative payment values,Checked,0
5,Invalid payment installments,Checked,2
6,Review scores outside 1–5,Checked,0
7,Lifecycle inconsistencies,Checked,1382


# 12. Analysis-Ready Copies

In [16]:
# Create copies for later analysis.
orders_clean = order_dates.copy()
geolocation_clean = geo_clean.copy()

# Keep the raw tables unchanged.
analysis_tables = tables.copy()

print("Analysis-ready order copy:", orders_clean.shape)
print("Deduplicated geolocation copy:", geolocation_clean.shape)

Analysis-ready order copy: (99441, 8)
Deduplicated geolocation copy: (738332, 5)


# 13. Save Safe Processed Outputs

In [17]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

geolocation_clean.to_csv(
    PROCESSED_DIR / "olist_geolocation_deduplicated.csv",
    index=False
)

orders_clean.to_csv(
    PROCESSED_DIR / "olist_orders_datetime_ready.csv",
    index=False
)

print("Processed files saved to:", PROCESSED_DIR)

Processed files saved to: E:\Interniship\HVIA - Data & AI Intern 2026\Tasks\Daily Tasks\Task 1\Olist_HVIA_Youssef_Atef_Tayh\data\processed


# 14. Data Quality Summary

The quality review identified:

- Missing values that require contextual treatment.
- Exact duplicate rows in the geolocation table.
- Lifecycle timestamp inconsistencies that should be interpreted carefully.
- Payment installment values below 1 that are flagged rather than silently corrected.
- No negative item prices, freight values, or payment values in the observed data.

The raw dataset remains unchanged.

The next stage can use the analysis-ready copies while keeping the original source data available for validation.

# 15. Next Stage: Exploratory Data Analysis

The next notebook will turn the cleaned structural understanding into business questions, comparisons, visualizations, and evidence-based insights.